# dense_embedding_exp2

본 노트북은 dense embedding cosine similarity 분석을 위한 exp2 데이터셋을 생성한다.

## exp2 분석 방식

exp2는 기준 문장과 공시 텍스트를 **섹션 단위**로 비교하는 방식이다.

즉, 기업 1개와 사업연도 1개 안에서 II, IV, VI, VIII, XI 섹션을 각각 따로 분리하여 분석 단위로 사용한다.

이후 다음 단계에서 각 섹션 텍스트와 E/S/G 기준 문장 간 dense embedding cosine similarity를 계산할 예정이다.

## exp2의 핵심 설계

- 비교 단위: 기업·연도·섹션
- 텍스트 구성: II, IV, VI, VIII, XI 섹션을 각각 별도 document로 유지
- 전처리 수준: dense embedding에 맞게 문장 의미를 최대한 보존
- 형태소 분석: 사용하지 않음
- 최종 컬럼: `exp2_section_text`


# 1. 라이브러리 및 경로 설정

In [1]:
# ============================================================
# 1. 라이브러리 및 경로 설정
# ============================================================

import json
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "data").exists():
            return candidate
    return Path.cwd()


PROJECT_ROOT = find_project_root()

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
ANALYSIS_DIR = PROCESSED_DIR / "analysis"

REPORT_JSON_DIR = PROCESSED_DIR / "dart_full_reports"
COMPANY_MASTER_PATH = RAW_DIR / "company_master.csv"

DENSE_DIR = ANALYSIS_DIR / "dense_embedding_compare"
DENSE_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("REPORT_JSON_DIR:", REPORT_JSON_DIR)
print("COMPANY_MASTER_PATH:", COMPANY_MASTER_PATH)
print("DENSE_DIR:", DENSE_DIR)

PROJECT_ROOT: c:\Users\duswl\OneDrive\바탕 화면\duswls3247\비정형 데이터 처리\Final
REPORT_JSON_DIR: c:\Users\duswl\OneDrive\바탕 화면\duswls3247\비정형 데이터 처리\Final\data\processed\dart_full_reports
COMPANY_MASTER_PATH: c:\Users\duswl\OneDrive\바탕 화면\duswls3247\비정형 데이터 처리\Final\data\raw\company_master.csv
DENSE_DIR: c:\Users\duswl\OneDrive\바탕 화면\duswls3247\비정형 데이터 처리\Final\data\processed\analysis\dense_embedding_compare


# 2. 분석 대상 섹션 설정

In [2]:
# ============================================================
# 2. 분석 대상 섹션 설정
# ============================================================

selected_sections = {
    "II": "II. 사업의 내용",
    "IV": "IV. 이사의 경영진단 및 분석의견",
    "VI": "VI. 이사회 등 회사의 기관에 관한 사항",
    "VIII": "VIII. 임원 및 직원 등에 관한 사항",
    "XI": "XI. 그 밖에 투자자 보호를 위하여 필요한 사항",
}

selected_romans = list(selected_sections.keys())

# 사업보고서 주요 목차 후보
# 섹션의 시작과 끝을 찾기 위해 전체 목차를 함께 사용
all_section_titles = {
    "I": ["회사의 개요"],
    "II": ["사업의 내용"],
    "III": ["재무에 관한 사항"],
    "IV": ["이사의 경영진단 및 분석의견", "이사의 경영진단", "경영진단 및 분석의견"],
    "V": ["회계감사인의 감사의견", "감사인의 감사의견"],
    "VI": ["이사회 등 회사의 기관에 관한 사항", "이사회 등 회사의 기관"],
    "VII": ["주주에 관한 사항"],
    "VIII": ["임원 및 직원 등에 관한 사항", "임원 및 직원"],
    "IX": ["계열회사 등에 관한 사항", "계열회사"],
    "X": ["대주주 등과의 거래내용", "대주주"],
    "XI": ["그 밖에 투자자 보호를 위하여 필요한 사항", "투자자 보호"],
    "XII": ["상세표", "부속명세서"],
}

roman_replace_map = {
    "Ⅰ": "I",
    "Ⅱ": "II",
    "Ⅲ": "III",
    "Ⅳ": "IV",
    "Ⅴ": "V",
    "Ⅵ": "VI",
    "Ⅶ": "VII",
    "Ⅷ": "VIII",
    "Ⅸ": "IX",
    "Ⅹ": "X",
    "Ⅺ": "XI",
    "Ⅻ": "XII",
}


def normalize_roman_text(text):
    text = str(text)
    for k, v in roman_replace_map.items():
        text = text.replace(k, v)
    return text


def normalize_space(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


print("분석 대상 섹션")
for k, v in selected_sections.items():
    print(k, ":", v)

분석 대상 섹션
II : II. 사업의 내용
IV : IV. 이사의 경영진단 및 분석의견
VI : VI. 이사회 등 회사의 기관에 관한 사항
VIII : VIII. 임원 및 직원 등에 관한 사항
XI : XI. 그 밖에 투자자 보호를 위하여 필요한 사항


# 3. JSON에서 보고서 전체 텍스트 구성

In [3]:
# ============================================================
# 3. JSON에서 보고서 전체 텍스트 구성
# ============================================================

def collect_text_blocks(obj):
    """
    JSON 내부에서 제목과 본문 텍스트를 순서대로 수집한다.
    """
    blocks = []

    title_keys = [
        "title", "section_title", "heading", "chapter", "toc", "section"
    ]

    text_keys = [
        "text", "full_text", "unit_text", "section_text",
        "content", "paragraph", "passage"
    ]

    if isinstance(obj, dict):
        # 제목 먼저 수집
        for key in title_keys:
            if key in obj and isinstance(obj[key], (str, int, float)):
                value = str(obj[key]).strip()
                if len(value) > 0:
                    blocks.append(value)

        # 본문 수집
        for key in text_keys:
            if key in obj and isinstance(obj[key], (str, int, float)):
                value = str(obj[key]).strip()
                if len(value) > 0:
                    blocks.append(value)

        # 하위 구조 탐색
        for value in obj.values():
            blocks.extend(collect_text_blocks(value))

    elif isinstance(obj, list):
        for item in obj:
            blocks.extend(collect_text_blocks(item))

    return blocks


def build_full_report_text(json_obj):
    """
    JSON에서 수집한 텍스트 블록을 하나의 보고서 텍스트로 결합한다.
    """
    blocks = collect_text_blocks(json_obj)

    # 너무 짧은 블록은 제외
    blocks = [str(x).strip() for x in blocks if len(str(x).strip()) >= 2]

    text = "\n".join(blocks)
    text = normalize_roman_text(text)

    # 목차 앞에 줄바꿈을 보강하여 섹션 탐색을 쉽게 함
    text = re.sub(
        r"\s+(I|II|III|IV|V|VI|VII|VIII|IX|X|XI|XII)\s*\.?\s+",
        r"\n\1. ",
        text
    )

    return text

# 4. 보고서 전체 텍스트에서 핵심 섹션 분리

- 보고서 전체 텍스트에서 I~XII 목차 위치를 찾는다.
- 그중 II, IV, VI, VIII, XI 섹션만 추출한다.
- 각 섹션은 다음 목차가 시작되기 전까지의 텍스트로 정의한다.

In [6]:
# ============================================================
# 4. 보고서 전체 텍스트에서 핵심 섹션 분리
# ============================================================

def find_section_headings(report_text):
    """
    보고서 전체 텍스트에서 I~XII 목차 heading 위치를 찾는다.
    start_group 오류 방지를 위해 중복 제거 방식을 안정적으로 수정한 버전.
    """
    report_text = normalize_roman_text(str(report_text))

    headings = []

    for roman, title_candidates in all_section_titles.items():
        for title in title_candidates:
            # 예: II. 사업의 내용 / II 사업의 내용
            pattern = rf"(?:^|\n|\s)\s*{roman}\s*\.?\s*{re.escape(title)}"

            for match in re.finditer(pattern, report_text, flags=re.IGNORECASE):
                headings.append({
                    "roman": roman,
                    "title": title,
                    "start": int(match.start()),
                    "end": int(match.end()),
                    "matched_text": match.group(0).strip()
                })

    if len(headings) == 0:
        return pd.DataFrame(columns=[
            "roman", "title", "start", "end", "matched_text"
        ])

    heading_df = pd.DataFrame(headings)

    # 위치 기준 정렬
    heading_df = heading_df.sort_values(["start", "end"]).reset_index(drop=True)

    # 너무 가까운 위치에서 반복 매칭된 heading 제거
    # start_group 대신 start_bucket 사용
    heading_df["start_bucket"] = (heading_df["start"] // 50) * 50

    heading_df = (
        heading_df
        .drop_duplicates(subset=["roman", "start_bucket"], keep="first")
        .drop(columns=["start_bucket"])
        .sort_values("start")
        .reset_index(drop=True)
    )

    return heading_df


def extract_selected_sections_from_report(report_text):
    """
    보고서 전체 텍스트에서 II, IV, VI, VIII, XI 섹션을 추출한다.
    """
    report_text = normalize_roman_text(str(report_text))

    heading_df = find_section_headings(report_text)

    if len(heading_df) == 0:
        return []

    extracted = []

    for i, row in heading_df.iterrows():
        roman = row["roman"]

        if roman not in selected_romans:
            continue

        start_pos = int(row["start"])

        # 다음 heading 시작 위치를 현재 섹션의 끝으로 사용
        next_headings = heading_df[heading_df["start"] > start_pos]

        if len(next_headings) > 0:
            end_pos = int(next_headings.iloc[0]["start"])
        else:
            end_pos = len(report_text)

        section_text = report_text[start_pos:end_pos]
        section_text = normalize_space(section_text)

        if len(section_text) < 50:
            continue

        extracted.append({
            "section_roman": roman,
            "section_clean": selected_sections[roman],
            "section_order": {
                "II": 2,
                "IV": 4,
                "VI": 6,
                "VIII": 8,
                "XI": 11,
            }[roman],
            "section_text": section_text,
            "section_text_chars": len(section_text),
            "matched_heading": row["matched_text"],
        })

    return extracted

# 5. DART JSON에서 섹션 단위 데이터 생성

- 각 JSON 파일을 읽는다.
- 보고서 전체 텍스트를 구성한다.
- II, IV, VI, VIII, XI 섹션을 직접 분리한다.
- 기업·연도·섹션 단위 데이터셋을 생성한다.

In [7]:
# ============================================================
# 5. DART JSON에서 섹션 단위 데이터 생성
# ============================================================

json_files = sorted(REPORT_JSON_DIR.glob("*.json"))

print("JSON 파일 수:", len(json_files))

if len(json_files) == 0:
    raise FileNotFoundError("dart_full_reports 폴더에 json 파일이 없습니다.")

rows = []

for path in json_files:
    file_name = path.name

    # 예: 005930_2024.json
    m = re.match(r"(\d{6})_(\d{4})\.json", file_name)

    if m:
        stock_code = m.group(1)
        fiscal_year = int(m.group(2))
    else:
        stock_code = ""
        fiscal_year = np.nan

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    report_text = build_full_report_text(data)

    extracted_sections = extract_selected_sections_from_report(report_text)

    for section in extracted_sections:
        analysis_unit = f"{str(stock_code).zfill(6)}x{int(fiscal_year)}"
        section_analysis_unit = f"{analysis_unit}_{section['section_roman']}"

        rows.append({
            "section_analysis_unit": section_analysis_unit,
            "analysis_unit": analysis_unit,
            "source_file": file_name,
            "stock_code": str(stock_code).zfill(6),
            "fiscal_year": int(fiscal_year),
            "esg_year": int(fiscal_year) + 1,
            "section_roman": section["section_roman"],
            "section_clean": section["section_clean"],
            "section_order": section["section_order"],
            "matched_heading": section["matched_heading"],
            "raw_section_text": section["section_text"],
            "section_text_chars": section["section_text_chars"],
        })

section_unit_df = pd.DataFrame(rows)

print("섹션 단위 데이터 수:", len(section_unit_df))

if len(section_unit_df) == 0:
    raise ValueError("섹션 단위 데이터가 생성되지 않았습니다. 목차 패턴을 확인하세요.")

print("\n섹션별 데이터 수:")
print(section_unit_df["section_clean"].value_counts())

print("\n기업-연도별 포함 섹션 개수 분포:")
print(
    section_unit_df
    .groupby("analysis_unit")["section_clean"]
    .nunique()
    .value_counts()
    .sort_index()
)

display(section_unit_df.head())

JSON 파일 수: 381
섹션 단위 데이터 수: 3134

섹션별 데이터 수:
section_clean
II. 사업의 내용                     768
VIII. 임원 및 직원 등에 관한 사항         696
XI. 그 밖에 투자자 보호를 위하여 필요한 사항    612
VI. 이사회 등 회사의 기관에 관한 사항        576
IV. 이사의 경영진단 및 분석의견            482
Name: count, dtype: int64

기업-연도별 포함 섹션 개수 분포:
section_clean
5    381
Name: count, dtype: int64


,section_analysis_unit,analysis_unit,source_file,stock_code,fiscal_year,esg_year,section_roman,section_clean,section_order,matched_heading,raw_section_text,section_text_chars
0,000020x2022_II,000020x2022,000020_2022.json,000020,2022,2023,II,II. 사업의 내용,2,II. 사업의 내용,II. 사업의 내용 1. 사업의 개요 1. 일반적인 사항지배기업인 연결실체는 제공하...,26053
1,000020x2022_IV,000020x2022,000020_2022.json,000020,2022,2023,IV,IV. 이사의 경영진단 및 분석의견,4,IV. 이사의 경영진단,IV. 이사의 경영진단 및 분석의견 1. 예측정보에 대한 주의사항 당사가 본 사업보...,5306
2,000020x2022_VI,000020x2022,000020_2022.json,000020,2022,2023,VI,VI. 이사회 등 회사의 기관에 관한 사항,6,VI. 이사회 등 회사의 기관,VI. 이사회 등 회사의 기관에 관한 사항 1. 이사회에 관한 사항 1. 이사회 구...,6798
3,000020x2022_VIII,000020x2022,000020_2022.json,000020,2022,2023,VIII,VIII. 임원 및 직원 등에 관한 사항,8,VIII. 임원 및 직원,VIII. 임원 및 직원 등에 관한 사항 1. 임원 및 직원 등의 현황 임원 현황 ...,4345
4,000020x2022_XI,000020x2022,000020_2022.json,000020,2022,2023,XI,XI. 그 밖에 투자자 보호를 위하여 필요한 사항,11,XI. 그 밖에 투자자 보호를 위하여 필요한 사항,XI. 그 밖에 투자자 보호를 위하여 필요한 사항 1. 공시내용 진행 및 변경사항 ...,903


# 6. 섹션 추출 품질 점검

In [8]:
# ============================================================
# 6. 섹션 추출 품질 점검
# ============================================================

section_count_by_unit = (
    section_unit_df
    .groupby("analysis_unit")["section_clean"]
    .nunique()
    .reset_index(name="section_count")
)

print("기업-연도별 섹션 개수 요약:")
display(section_count_by_unit["section_count"].describe())

print("\n기업-연도별 섹션 개수 분포:")
print(section_count_by_unit["section_count"].value_counts().sort_index())

print("\n섹션별 추출 개수:")
print(section_unit_df["section_clean"].value_counts())

# 5개 미만으로 추출된 기업-연도 확인
under_extracted = section_count_by_unit[section_count_by_unit["section_count"] < 5]

print("\n5개 미만 섹션이 추출된 기업-연도 수:", len(under_extracted))

display(under_extracted.head(20))

기업-연도별 섹션 개수 요약:


count    381.0
mean       5.0
std        0.0
min        5.0
25%        5.0
50%        5.0
75%        5.0
max        5.0
Name: section_count, dtype: float64


기업-연도별 섹션 개수 분포:
section_count
5    381
Name: count, dtype: int64

섹션별 추출 개수:
section_clean
II. 사업의 내용                     768
VIII. 임원 및 직원 등에 관한 사항         696
XI. 그 밖에 투자자 보호를 위하여 필요한 사항    612
VI. 이사회 등 회사의 기관에 관한 사항        576
IV. 이사의 경영진단 및 분석의견            482
Name: count, dtype: int64

5개 미만 섹션이 추출된 기업-연도 수: 0


,analysis_unit,section_count


# 7. ESG 등급 데이터 병합

In [9]:
# ============================================================
# 7. ESG 등급 데이터 병합
# ============================================================

company_master = pd.read_csv(COMPANY_MASTER_PATH, dtype={"stock_code": str})
company_master.columns = [str(c).strip() for c in company_master.columns]

company_master["stock_code"] = (
    company_master["stock_code"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(6)
)

company_master["fiscal_year"] = company_master["fiscal_year"].astype(int)

if "esg_year" in company_master.columns:
    company_master["esg_year"] = company_master["esg_year"].astype(int)
else:
    company_master["esg_year"] = company_master["fiscal_year"] + 1

grade_map = {
    "D": 0,
    "C": 1,
    "B": 2,
    "B+": 3,
    "A": 4,
    "A+": 5,
    "S": 6,
}

if "esg_grade_num" not in company_master.columns:
    company_master["esg_grade"] = company_master["esg_grade"].astype(str).str.strip()
    company_master["esg_grade_num"] = company_master["esg_grade"].map(grade_map)

grade_cols = [
    "stock_code",
    "fiscal_year",
    "esg_year",
    "esg_grade",
    "esg_grade_num",
]

for col in ["company_name", "회사이름", "e_grade", "s_grade", "g_grade"]:
    if col in company_master.columns:
        grade_cols.append(col)

analysis_df = pd.merge(
    section_unit_df,
    company_master[grade_cols].drop_duplicates(),
    on=["stock_code", "fiscal_year", "esg_year"],
    how="left"
)

print("병합 후 데이터 수:", len(analysis_df))
print("ESG 등급 결측 수:", analysis_df["esg_grade"].isna().sum())

display(analysis_df.head())

병합 후 데이터 수: 3134
ESG 등급 결측 수: 0


,section_analysis_unit,analysis_unit,source_file,stock_code,fiscal_year,esg_year,section_roman,section_clean,section_order,matched_heading,raw_section_text,section_text_chars,esg_grade,esg_grade_num,company_name,e_grade,s_grade,g_grade
0,000020x2022_II,000020x2022,000020_2022.json,000020,2022,2023,II,II. 사업의 내용,2,II. 사업의 내용,II. 사업의 내용 1. 사업의 개요 1. 일반적인 사항지배기업인 연결실체는 제공하...,26053,C,1,동화약품,C,B,C
1,000020x2022_IV,000020x2022,000020_2022.json,000020,2022,2023,IV,IV. 이사의 경영진단 및 분석의견,4,IV. 이사의 경영진단,IV. 이사의 경영진단 및 분석의견 1. 예측정보에 대한 주의사항 당사가 본 사업보...,5306,C,1,동화약품,C,B,C
2,000020x2022_VI,000020x2022,000020_2022.json,000020,2022,2023,VI,VI. 이사회 등 회사의 기관에 관한 사항,6,VI. 이사회 등 회사의 기관,VI. 이사회 등 회사의 기관에 관한 사항 1. 이사회에 관한 사항 1. 이사회 구...,6798,C,1,동화약품,C,B,C
3,000020x2022_VIII,000020x2022,000020_2022.json,000020,2022,2023,VIII,VIII. 임원 및 직원 등에 관한 사항,8,VIII. 임원 및 직원,VIII. 임원 및 직원 등에 관한 사항 1. 임원 및 직원 등의 현황 임원 현황 ...,4345,C,1,동화약품,C,B,C
4,000020x2022_XI,000020x2022,000020_2022.json,000020,2022,2023,XI,XI. 그 밖에 투자자 보호를 위하여 필요한 사항,11,XI. 그 밖에 투자자 보호를 위하여 필요한 사항,XI. 그 밖에 투자자 보호를 위하여 필요한 사항 1. 공시내용 진행 및 변경사항 ...,903,C,1,동화약품,C,B,C


# 8. 섹션 단위 dense embedding용 기본 정제

dense embedding 방식에서는 문장 의미 보존이 중요하므로 형태소 분석은 적용하지 않는다.

이 단계에서는 다음 정도의 기본 정제만 수행한다.

- 공백 정리
- 줄바꿈 정리
- 깨진 특수문자 일부 제거
- 과도한 기호 제거
- 문장 구조는 최대한 유지

결과 컬럼은 `exp2_section_text`이다.

In [10]:
# ============================================================
# 8. 섹션 단위 dense embedding용 기본 정제
# ============================================================

def clean_for_dense_section(text):
    text = str(text)

    # HTML 태그처럼 보이는 것 제거
    text = re.sub(r"<[^>]+>", " ", text)

    # 분석에 크게 필요 없는 특수문자 제거
    # 단, 문장부호 일부는 의미 흐름을 위해 유지
    text = re.sub(r"[^\w가-힣\s.,;:!?%()\-]", " ", text)

    # 로마숫자 특수문자 정리
    text = normalize_roman_text(text)

    # 공백 정리
    text = re.sub(r"\s+", " ", text).strip()

    return text


analysis_df["exp2_section_text"] = analysis_df["raw_section_text"].apply(
    clean_for_dense_section
)

analysis_df["exp2_section_text_chars"] = (
    analysis_df["exp2_section_text"]
    .fillna("")
    .astype(str)
    .str.len()
)

print("섹션 단위 정제 텍스트 생성 완료")

display(
    analysis_df[
        [
            "section_analysis_unit",
            "section_clean",
            "raw_section_text",
            "exp2_section_text",
            "exp2_section_text_chars",
        ]
    ].head(10)
)

섹션 단위 정제 텍스트 생성 완료


,section_analysis_unit,section_clean,raw_section_text,exp2_section_text,exp2_section_text_chars
0,000020x2022_II,II. 사업의 내용,II. 사업의 내용 1. 사업의 개요 1. 일반적인 사항지배기업인 연결실체는 제공하...,II. 사업의 내용 1. 사업의 개요 1. 일반적인 사항지배기업인 연결실체는 제공하...,25949
1,000020x2022_IV,IV. 이사의 경영진단 및 분석의견,IV. 이사의 경영진단 및 분석의견 1. 예측정보에 대한 주의사항 당사가 본 사업보...,IV. 이사의 경영진단 및 분석의견 1. 예측정보에 대한 주의사항 당사가 본 사업보...,5282
2,000020x2022_VI,VI. 이사회 등 회사의 기관에 관한 사항,VI. 이사회 등 회사의 기관에 관한 사항 1. 이사회에 관한 사항 1. 이사회 구...,VI. 이사회 등 회사의 기관에 관한 사항 1. 이사회에 관한 사항 1. 이사회 구...,6779
3,000020x2022_VIII,VIII. 임원 및 직원 등에 관한 사항,VIII. 임원 및 직원 등에 관한 사항 1. 임원 및 직원 등의 현황 임원 현황 ...,VIII. 임원 및 직원 등에 관한 사항 1. 임원 및 직원 등의 현황 임원 현황 ...,4342
4,000020x2022_XI,XI. 그 밖에 투자자 보호를 위하여 필요한 사항,XI. 그 밖에 투자자 보호를 위하여 필요한 사항 1. 공시내용 진행 및 변경사항 ...,XI. 그 밖에 투자자 보호를 위하여 필요한 사항 1. 공시내용 진행 및 변경사항 ...,903
5,000020x2023_II,II. 사업의 내용,II. 사업의 내용 1. 사업의 개요 1. 일반적인 사항지배기업인 연결실체는 제공하...,II. 사업의 내용 1. 사업의 개요 1. 일반적인 사항지배기업인 연결실체는 제공하...,26467
6,000020x2023_IV,IV. 이사의 경영진단 및 분석의견,IV. 이사의 경영진단 및 분석의견 1. 예측정보에 대한 주의사항 당사가 본 사업보...,IV. 이사의 경영진단 및 분석의견 1. 예측정보에 대한 주의사항 당사가 본 사업보...,5288
7,000020x2023_VI,VI. 이사회 등 회사의 기관에 관한 사항,VI. 이사회 등 회사의 기관에 관한 사항 1. 이사회에 관한 사항 1. 이사회 구...,VI. 이사회 등 회사의 기관에 관한 사항 1. 이사회에 관한 사항 1. 이사회 구...,6733
8,000020x2023_VIII,VIII. 임원 및 직원 등에 관한 사항,VIII. 임원 및 직원 등에 관한 사항 1. 임원 및 직원 등의 현황 임원 현황 ...,VIII. 임원 및 직원 등에 관한 사항 1. 임원 및 직원 등의 현황 임원 현황 ...,4208
9,000020x2023_XI,XI. 그 밖에 투자자 보호를 위하여 필요한 사항,XI. 그 밖에 투자자 보호를 위하여 필요한 사항 1. 공시내용 진행 및 변경사항 ...,XI. 그 밖에 투자자 보호를 위하여 필요한 사항 1. 공시내용 진행 및 변경사항 ...,1177


# 9. exp2 데이터 점검

In [11]:
# ============================================================
# 9. exp2 데이터 점검
# ============================================================

print("전체 섹션 단위 데이터 수:", len(analysis_df))
print("기업 수:", analysis_df["stock_code"].nunique())

print("\n사업연도 분포:")
print(analysis_df["fiscal_year"].value_counts().sort_index())

print("\n섹션별 데이터 수:")
print(analysis_df["section_clean"].value_counts())

print("\n기업-연도별 포함 섹션 개수 분포:")
print(
    analysis_df
    .groupby("analysis_unit")["section_clean"]
    .nunique()
    .value_counts()
    .sort_index()
)

length_summary = analysis_df[
    ["section_text_chars", "exp2_section_text_chars"]
].describe().T

display(length_summary)

sample_cols = [
    "section_analysis_unit",
    "analysis_unit",
    "company_name" if "company_name" in analysis_df.columns else "stock_code",
    "fiscal_year",
    "esg_year",
    "esg_grade",
    "section_clean",
    "exp2_section_text",
]

sample_cols = [c for c in sample_cols if c in analysis_df.columns]

display(analysis_df[sample_cols].head(15))

전체 섹션 단위 데이터 수: 3134
기업 수: 127

사업연도 분포:
fiscal_year
2022    1040
2023    1039
2024    1055
Name: count, dtype: int64

섹션별 데이터 수:
section_clean
II. 사업의 내용                     768
VIII. 임원 및 직원 등에 관한 사항         696
XI. 그 밖에 투자자 보호를 위하여 필요한 사항    612
VI. 이사회 등 회사의 기관에 관한 사항        576
IV. 이사의 경영진단 및 분석의견            482
Name: count, dtype: int64

기업-연도별 포함 섹션 개수 분포:
section_clean
5    381
Name: count, dtype: int64


,count,mean,std,min,25%,50%,75%,max
section_text_chars,3134.0,11181.248883,18435.837990,50.0,321.0,5628.5,14058.75,219367.0
exp2_section_text_chars,3134.0,11118.154116,18346.486774,50.0,321.0,5565.5,14004.50,218699.0


,section_analysis_unit,analysis_unit,company_name,fiscal_year,esg_year,esg_grade,section_clean,exp2_section_text
0,000020x2022_II,000020x2022,동화약품,2022,2023,C,II. 사업의 내용,II. 사업의 내용 1. 사업의 개요 1. 일반적인 사항지배기업인 연결실체는 제공하...
1,000020x2022_IV,000020x2022,동화약품,2022,2023,C,IV. 이사의 경영진단 및 분석의견,IV. 이사의 경영진단 및 분석의견 1. 예측정보에 대한 주의사항 당사가 본 사업보...
2,000020x2022_VI,000020x2022,동화약품,2022,2023,C,VI. 이사회 등 회사의 기관에 관한 사항,VI. 이사회 등 회사의 기관에 관한 사항 1. 이사회에 관한 사항 1. 이사회 구...
3,000020x2022_VIII,000020x2022,동화약품,2022,2023,C,VIII. 임원 및 직원 등에 관한 사항,VIII. 임원 및 직원 등에 관한 사항 1. 임원 및 직원 등의 현황 임원 현황 ...
4,000020x2022_XI,000020x2022,동화약품,2022,2023,C,XI. 그 밖에 투자자 보호를 위하여 필요한 사항,XI. 그 밖에 투자자 보호를 위하여 필요한 사항 1. 공시내용 진행 및 변경사항 ...
5,000020x2023_II,000020x2023,동화약품,2023,2024,C,II. 사업의 내용,II. 사업의 내용 1. 사업의 개요 1. 일반적인 사항지배기업인 연결실체는 제공하...
6,000020x2023_IV,000020x2023,동화약품,2023,2024,C,IV. 이사의 경영진단 및 분석의견,IV. 이사의 경영진단 및 분석의견 1. 예측정보에 대한 주의사항 당사가 본 사업보...
7,000020x2023_VI,000020x2023,동화약품,2023,2024,C,VI. 이사회 등 회사의 기관에 관한 사항,VI. 이사회 등 회사의 기관에 관한 사항 1. 이사회에 관한 사항 1. 이사회 구...
8,000020x2023_VIII,000020x2023,동화약품,2023,2024,C,VIII. 임원 및 직원 등에 관한 사항,VIII. 임원 및 직원 등에 관한 사항 1. 임원 및 직원 등의 현황 임원 현황 ...
9,000020x2023_XI,000020x2023,동화약품,2023,2024,C,XI. 그 밖에 투자자 보호를 위하여 필요한 사항,XI. 그 밖에 투자자 보호를 위하여 필요한 사항 1. 공시내용 진행 및 변경사항 ...


# 10. exp2 결과 저장


In [12]:
# ============================================================
# 10. exp2 결과 저장
# ============================================================

output_path = DENSE_DIR / "dense_embedding_exp2_section_unit.csv"

save_cols = []

for col in [
    "section_analysis_unit",
    "analysis_unit",
    "stock_code",
    "company_name",
    "회사이름",
    "fiscal_year",
    "esg_year",
    "esg_grade",
    "esg_grade_num",
    "section_roman",
    "section_clean",
    "section_order",
    "matched_heading",
    "section_text_chars",
    "raw_section_text",
    "exp2_section_text",
    "exp2_section_text_chars",
]:
    if col in analysis_df.columns and col not in save_cols:
        save_cols.append(col)

analysis_df[save_cols].to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", output_path)
print("저장 컬럼:")
print(save_cols)

저장 완료: c:\Users\duswl\OneDrive\바탕 화면\duswls3247\비정형 데이터 처리\Final\data\processed\analysis\dense_embedding_compare\dense_embedding_exp2_section_unit.csv
저장 컬럼:
['section_analysis_unit', 'analysis_unit', 'stock_code', 'company_name', 'fiscal_year', 'esg_year', 'esg_grade', 'esg_grade_num', 'section_roman', 'section_clean', 'section_order', 'matched_heading', 'section_text_chars', 'raw_section_text', 'exp2_section_text', 'exp2_section_text_chars']
